# Lektion 11 - Agent-till-agent (A2A) protokoll


## Installation


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Vad är A2A-protokollet?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agenter publicerar ett *Agentkort* som beskriver deras möjligheter, vilket gör det
  enkelt för andra agenter (eller orkestratorer) att hitta rätt specialist för en uppgift.
- **Message Passing** – Agenter utbyter strukturerade meddelanden via ett gemensamt protokoll, så att en
  förfrågan från en agent kan förstås och uppfyllas av en annan oavsett dess interna implementering.
- **Task Lifecycle** – A2A definierar tillstånd som *submitted*, *working*, *completed*, och
  *failed*, vilket ger orkestratorn full insyn i hur en delegerad uppgift utvecklas.

I den här lektionen simulerar vi A2A-liknande samarbete genom att koppla samman tre specialiserade reseagenter
i ett arbetsflöde där varje agent bidrar med sin expertis och för vidare resultatet till nästa.


## Skapa specialiserade reseagenter


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Samarbete mellan flera agenter via arbetsflöde

Vi kopplar ihop de tre agenterna i ett sekventiellt arbetsflöde som speglar A2A-meddelandeöverföring:

1. **CurrencyExchangeAgent** tar emot användarens förfrågan och ger valutavägledning.
2. **ActivityPlannerAgent** tar emot den berikade kontexten och lägger till aktivitetsrekommendationer.
3. **TravelManagerAgent** sammanställer båda indata till en slutlig reseöversikt.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Förstå A2A i produktion

I en produktionsmiljö öppnar A2A-protokollet upp kraftfulla scenarier över tjänster:

| Funktion | Beskrivning |
|---|---|
| **Interoperabilitet mellan ramverk** | En agent byggd med ett ramverk kan delegera uppgifter till en agent byggd med vilket annat A2A-kompatibelt ramverk som helst, vilket möjliggör verklig interoperabilitet mellan organisationer. |
| **Tjänstegränser** | Agenter kan leva i separata mikrotjänster, molnregioner eller till och med olika organisationer samtidigt som de samarbetar sömlöst. |
| **Dynamisk upptäckt** | En orkestrator kan fråga ett Agent Card-registre vid körning för att hitta den mest lämpliga specialisten för en viss deluppgift. |
| **Streaming & push-notiser** | A2A stödjer Server-Sent Events (SSE) för realtidsuppdateringar av framsteg och push-notiser för långkörande uppgifter. |

Arbetsflödet vi byggde ovan är en förenklad, in-process-version av detta mönster. I en verklig driftsättning skulle varje agent exponera en HTTP-endpoint, publicera ett Agent Card, och kommunicera via A2A JSON-RPC-protokollet.


## Summary

I den här lektionen lärde du dig:

1. **Vad A2A-protokollet är** — en öppen standard för agent-till-agent-upptäckt, meddelandehantering,
   och uppgiftshantering.
2. **Hur man skapar specialiserade agenter** — en agent för valutaväxling, en agent för aktivitetsplanering,
   och en orkestrator för resehantering.
3. **Hur man kopplar agenter till ett arbetsflöde** — genom att använda `WorkflowBuilder` för att modellera sekventiell
   meddelandeöverföring mellan agenter.
4. **Hur A2A fungerar i produktion** — möjliggör samarbete över ramverk och tjänster
   med dynamisk upptäckt och strömmande uppdateringar.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Ansvarsfriskrivning:
Detta dokument har översatts med hjälp av AI-översättningstjänsten Co-op Translator (https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet bör du vara medveten om att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på sitt ursprungliga språk bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
